In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_7_try_1.pickle

In [ ]:
%%cudf.pandas.profile
### cell 8 ###

# re-introduce the question and x-axis title variables
question_of_interest_cell20 = 'What is your age (# years)?'
title_for_x_axis_cell20 = ''

# compute normalized value counts as percentages
def count_then_return_percent_20(df, col):
    pct = df[col].value_counts(dropna=False, normalize=True).sort_index()
    return (pct * 100).round(1)

# combine multiple years into one DataFrame, adding year and x-axis columns
def combine_subset_of_data_from_multiple_years_20(question, x_axis, include_2017=False):
    sources = [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10),
    ]
    if include_2017:
        sources.insert(0, ('2017', responses_df_2017))

    dfs = []
    for year, df in sources:
        # get a Series of percentages, rename it
        s = count_then_return_percent_20(df, question).rename('percentage')
        # to_frame preserves the GPU-backed DataFrame
        df_pct = s.to_frame()
        # lift the index into an x-axis column
        df_pct[x_axis] = df_pct.index
        # add the year column
        df_pct['year'] = year
        dfs.append(df_pct)

    # concat all years, resetting the index
    df_combined = pd.concat(dfs, ignore_index=True)
    # ensure final column order
    return df_combined[['percentage', 'year', x_axis]]

# apply the age-group replacement in-place on GPU
responses_df_2018_cell10[question_of_interest_cell20] = \
    responses_df_2018_cell10[question_of_interest_cell20].replace(['70-79', '80+'], '70+')

# build the combined DataFrame (2017 excluded by default)
age_df_combined = combine_subset_of_data_from_multiple_years_20(
    question_of_interest_cell20,
    title_for_x_axis_cell20
)
age_df_combined.info()